In [1]:
import os
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import ast
import pandas as pd
import string

In [2]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common room',
    (0xFF, 0xA5, 0x00): 'master room',
    (0xEE, 0xE8, 0xAA): 'living room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

In [3]:
base_dir = Path("../..")
json_path = Path("difficult_dataset.json")      # or "difficult_dataset.json"
img_dir = base_dir / "data" / "floorplan_image"
txt_dir = base_dir / "annotation" / "human_annotated_tags"  # optional (kept)
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)




In [5]:
# --- Load JSON entries -------------------------------------------------------
# Expects a list of dicts with keys: dataset, base_cluster, other_cluster, group, outlier_id
with open(json_path, "r") as f:
    entries = json.load(f)

# --- Font for subtitles ------------------------------------------------------
try:
    font = ImageFont.truetype("arial.ttf", size=16)
except IOError:
    font = ImageFont.load_default()

subtitle_height = 24  # space for ID subtitles

def draw_outline(draw, xy, width=4):
    """Draw a rectangular outline with specified 'width' pixels."""
    x0, y0, x1, y1 = xy
    for i in range(width):
        draw.rectangle((x0 - i, y0 - i, x1 + i, y1 + i), outline=(220, 0, 0))

# --- Iterate groups ----------------------------------------------------------
num_rendered = 0

for idx, entry in enumerate(entries, start=1):
    # Support both dict and legacy list format
    if isinstance(entry, dict) and "group" in entry:
        group_ids = entry["group"]
        outlier_id = entry.get("outlier_id")
        dataset_tag = entry.get("dataset", "unknown")
    elif isinstance(entry, list):
        group_ids = entry
        outlier_id = None
        dataset_tag = "unknown"
    else:
        # Skip any malformed item
        continue

    # Load images
    images = []
    for img_id in group_ids:
        img_path = img_dir / f"{img_id}.png"
        if not img_path.exists():
            raise FileNotFoundError(f"Image file not found: {img_path}")
        images.append((img_id, Image.open(img_path).convert("RGB")))

    # Layout
    widths, heights = zip(*(im.size for _, im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Canvas
    canvas = Image.new("RGB", (total_width, max_height + subtitle_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    # Paste images, draw subtitles, and outline outlier (if present)
    x_offset = 0
    for img_id, im in images:
        canvas.paste(im, (x_offset, 0))

        # Subtitle text size
        try:
            text_w, text_h = font.getsize(img_id)
        except AttributeError:
            bbox = draw.textbbox((0, 0), img_id, font=font)
            text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]

        text_x = x_offset + (im.width - text_w) // 2
        text_y = max_height + (subtitle_height - text_h) // 2
        draw.text((text_x, text_y), img_id, fill=(0, 0, 0), font=font)

        # Highlight outlier with a red rectangle & small label
        if outlier_id is not None and img_id == outlier_id:
            draw_outline(draw, (x_offset, 0, x_offset + im.width - 1, im.height - 1), width=5)
            label = "OUTLIER"
            try:
                lw, lh = font.getsize(label)
            except AttributeError:
                lb = draw.textbbox((0, 0), label, font=font)
                lw, lh = lb[2] - lb[0], lb[3] - lb[1]
            pad = 4
            # small white box for readability
            draw.rectangle((x_offset + 6, 6, x_offset + 6 + lw + 2 * pad, 6 + lh + 2 * pad), fill=(255, 255, 255))
            draw.text((x_offset + 6 + pad, 6 + pad), label, fill=(220, 0, 0), font=font)

        x_offset += im.width

    # Save
    img_out_path = out_dir / f"{dataset_tag}_example_{idx}.png"
    canvas.save(img_out_path)
    num_rendered += 1

    # --- Optional: write descriptions (kept as a switch; off by default) -----
    # write_txt = False
    # if write_txt:
    #     txt_out_path = out_dir / f"{dataset_tag}_example_{idx}.txt"
    #     with open(txt_out_path, "w") as txt_out:
    #         for img_id, _ in images:
    #             tag_file = txt_dir / f"{img_id}.txt"
    #             if not tag_file.exists():
                #     raise FileNotFoundError(f"Tag file not found: {tag_file}")
    #             with open(tag_file, "r") as tf:
    #                 desc = tf.read().strip()
    #             txt_out.write(f"ID: {img_id}, description: {desc}\n\n")

print(f"Generated {num_rendered} example image(s) in '{out_dir.resolve()}' from '{json_path.name}'")

Generated 100 example image(s) in '/ssd2/aliceliu/surprise_sae/planscape/examples/complex/examples_output' from 'easy_dataset.json'


In [36]:
input_folder  = 'examples_same_second'
output_folder = os.path.join(input_folder, 'combined')
os.makedirs(output_folder, exist_ok=True)

# Find every file that ends with '_2.png'
pattern = os.path.join(input_folder, '*_2.png')
for suffix_path in glob.glob(pattern):
    # Derive the base filename by stripping off '_2' before the extension
    folder, suffix_fn = os.path.split(suffix_path)
    stem = suffix_fn[:-6]  # removes the trailing "_2.png"
    base_fn = stem + '.png'
    base_path = os.path.join(folder, base_fn)

    if not os.path.exists(base_path):
        print(f"No base image found for {suffix_fn}, skipping.")
        continue

    # Open both images (they must be same size)
    img_base   = Image.open(base_path)
    img_suffix = Image.open(suffix_path)
    w, h       = img_base.size

    # Stack base on top of suffix
    combined = Image.new('RGB', (w, 2*h))
    combined.paste(img_base,   (0, 0))
    combined.paste(img_suffix, (0, h))

    # Save
    out_fn = f"{stem}_combined.png"
    combined.save(os.path.join(output_folder, out_fn))



No base image found for example_2.png, skipping.


In [6]:
base_dir = Path("../..")
csv_path = "groups_same_second_results.csv"
img_dir = base_dir / "data" / "floorplan_image"
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)

# Try to load a scalable font at 24px
font_size = 24
for font_name in ("arial.ttf", "DejaVuSans.ttf"):
    try:
        font = ImageFont.truetype(font_name, size=font_size)
        break
    except IOError:
        font = None
if font is None:
    # last resort: default (will ignore size)
    font = ImageFont.load_default()
    print("Warning: falling back to default font; letter size may not change.")

subtitle_height = font_size + 8  # give a bit of padding

# Read groups
df = pd.read_csv(csv_path)
entries = df['options'].apply(lambda s: ast.literal_eval(s))

# Prepare A, B, C… labels
letters = list(string.ascii_uppercase)

for idx, entry in enumerate(entries, start=1):
    images = []

    # Load images for this group
    for img_id in entry:
        img_path = img_dir / f"{img_id}.png"
        if not img_path.exists():
            raise FileNotFoundError(f"Image file not found: {img_path}")
        images.append(Image.open(img_path))

    # Compute canvas size
    widths, heights = zip(*(im.size for im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Create canvas
    canvas = Image.new("RGB", (total_width, max_height + subtitle_height), "white")
    draw = ImageDraw.Draw(canvas)

    # Paste & subtitle
    x = 0
    for i, im in enumerate(images):
        canvas.paste(im, (x, 0))
        label = letters[i]
        # measure
        if hasattr(font, "getsize"):
            w, h = font.getsize(label)
        else:
            bx0, by0, bx1, by1 = draw.textbbox((0,0), label, font=font)
            w, h = bx1 - bx0, by1 - by0
        tx = x + (im.width - w) // 2
        ty = max_height + (subtitle_height - h) // 2
        draw.text((tx, ty), label, fill="black", font=font)
        x += im.width

    # Save
    canvas.save(out_dir / f"example_{idx}_2.png")

print(f"Generated {len(entries)} example image groups in '{out_dir.resolve()}'")

Generated 100 example image groups in '/home/airlay88/planscape/examples/complex/examples_output'


In [5]:
def add_legend_tight(
    image: Image.Image,
    mapping: dict,
    legend_gap: int = 75,
    font_size: int = 16
) -> Image.Image:
    swatch  = 20
    pad_x   = 5
    pad_y   = 2
    pad_mid = legend_gap

    # try to load a truetype font at the requested size, fallback to default
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
    except IOError:
        font = ImageFont.load_default()

    entries = list(mapping.items())

    # measure text widths
    dummy     = Image.new("RGB", (1,1))
    draw0     = ImageDraw.Draw(dummy)
    text_widths = [
        draw0.textbbox((0,0), label, font=font)[2]
        for _, label in entries
    ]

    # compute legend dims
    entry_widths   = [swatch + pad_x + w for w in text_widths]
    total_legend_w = sum(entry_widths) + pad_x * (len(entries) + 1)
    legend_h       = swatch + 2 * pad_y

    # render legend
    legend = Image.new("RGB", (total_legend_w, legend_h), "white")
    draw   = ImageDraw.Draw(legend)

    x = pad_x
    for (color, label), w in zip(entries, text_widths):
        draw.rectangle([x, pad_y, x + swatch, pad_y + swatch],
                       fill=color, outline="black")
        draw.text((x + swatch + pad_x, pad_y),
                  label, fill="black", font=font)
        x += swatch + pad_x + w + pad_x

    # composite under the image
    out_w = max(image.width, total_legend_w)
    out_h = image.height + pad_mid + legend_h
    combined = Image.new("RGB", (out_w, out_h), "white")
    combined.paste(image, ((out_w - image.width)//2, 0))
    combined.paste(legend, ((out_w - total_legend_w)//2, image.height + pad_mid))

    return combined

In [9]:
from PIL import Image, ImageDraw, ImageFont

def add_legend_tight_boxed(
    image: Image.Image,
    mapping: dict,
    legend_gap: int = 75,
    font_size: int = 16,
    title: str = "Legend",
    title_font_size: int = None,
    box_pad: int = 8,              # padding inside the box
    title_gap: int = 6,            # space between title and entries
    box_border: int = 2,           # border thickness
    box_bg: str = "white",
    box_border_color: str = "black",
    round_radius: int = 10         # set to 0 for square corners
) -> Image.Image:
    """
    Renders a single-row legend wrapped in a labeled box under the image.
    """

    swatch  = 20
    pad_x   = 5
    pad_y   = 2
    pad_mid = legend_gap

    # fonts
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
        tfont = ImageFont.truetype("DejaVuSans.ttf", title_font_size or (font_size + 2))
    except IOError:
        font = ImageFont.load_default()
        tfont = ImageFont.load_default()

    entries = list(mapping.items())

    # --- measure text widths for entries ---
    dummy = Image.new("RGB", (1, 1))
    d0 = ImageDraw.Draw(dummy)

    text_widths = [d0.textbbox((0, 0), label, font=font)[2] for _, label in entries]
    entry_widths   = [swatch + pad_x + w for w in text_widths]
    total_legend_w = sum(entry_widths) + pad_x * (len(entries) + 1)
    legend_h       = swatch + 2 * pad_y

    # --- measure title ---
    title_bbox = d0.textbbox((0, 0), title, font=tfont)
    title_w = title_bbox[2] - title_bbox[0]
    title_h = title_bbox[3] - title_bbox[1]

    # --- compute box dims ---
    inner_w = max(total_legend_w, title_w)
    inner_h = title_h + title_gap + legend_h
    box_w   = inner_w + 2 * box_pad
    box_h   = inner_h + 2 * box_pad

    # --- render boxed legend ---
    panel = Image.new("RGB", (box_w, box_h), box_bg)
    draw  = ImageDraw.Draw(panel)

    # border (rounded if available)
    rect_xy = [0.5, 0.5, box_w - 0.5, box_h - 0.5]  # half-pixel aligns stroke nicely
    if round_radius > 0 and hasattr(draw, "rounded_rectangle"):
        draw.rounded_rectangle(rect_xy, radius=round_radius, outline=box_border_color, width=box_border, fill=box_bg)
    else:
        draw.rectangle(rect_xy, outline=box_border_color, width=box_border, fill=box_bg)

    # title (centered)
    title_x = (box_w - title_w) // 2
    title_y = box_pad
    draw.text((title_x, title_y), title, fill="black", font=tfont)

    # entries row
    x = (box_w - total_legend_w) // 2
    y_top = box_pad + title_h + title_gap
    for (color, label), w in zip(entries, text_widths):
        # swatch
        draw.rectangle([x, y_top + pad_y, x + swatch, y_top + pad_y + swatch],
                       fill=color, outline="black")
        # label
        draw.text((x + swatch + pad_x, y_top + pad_y), label, fill="black", font=font)
        x += swatch + pad_x + w + pad_x

    # --- composite under the image ---
    out_w = max(image.width, box_w)
    out_h = image.height + pad_mid + box_h
    combined = Image.new("RGB", (out_w, out_h), "white")
    combined.paste(image, ((out_w - image.width) // 2, 0))
    combined.paste(panel, ((out_w - box_w) // 2, image.height + pad_mid))

    return combined


In [5]:
def draw_5x256_boxes(image, box_color="gray", box_width=3):
    """
    Draws five 256x256 gray boxes at (0,0), (256,0), (512,0), (768,0), (1024,0).
    Designed for 1280x288 images where the bottom ~32px holds labels.
    """
    W, H = image.size
    n, s = 5, 256
    total = n * s
    x0 = 0 if W == total else max(0, (W - total) // 2)
    y0 = 0  # tiles start at the top

    out = image.copy()
    draw = ImageDraw.Draw(out)

    for i in range(n):
        x = x0 + i * s
        y = y0
        draw.rectangle([x, y, x + s, y + s], outline=box_color, width=box_width)

    return out

In [9]:
in_dir = "./randomized_options/examples_diff_first"
out_dir = "./randomized_options/examples_diff_first_wlegend_onlyoptionbound"

In [4]:
for fn in os.listdir(in_dir):
    img_path = os.path.join(in_dir, fn)
    
    im = add_legend_tight_boxed(Image.open(img_path), COLOR_MAPPING, title="Color-Coding of Room Types")
    output_path = os.path.join(out_dir, fn)

    im.save(output_path, format="PNG")

NameError: name 'add_legend_tight_boxed' is not defined

In [10]:
for fn in os.listdir(in_dir):
    img_path = os.path.join(in_dir, fn)
    im = Image.open(img_path)

    # draw the 5 fixed 256×256 boxes
    im = draw_5x256_boxes(im)

    # add your boxed legend
    # im = add_legend_tight_boxed(im, COLOR_MAPPING, title="Color-Coding of Room Types")

    output_path = os.path.join(out_dir, fn)
    im.save(output_path, format="PNG")